# The neuron & network layers

_Deep Learning_

**A neuron is a dot product plus a bump, then a squish. Stack many and you get a brain-ish machine.**

A neuron is the tiny building block of a neural network.

     It takes some input numbers. It multiplies each by a weight. It adds them up. It adds one more number called a bias. Then it squishes the result through an activation function.

---

This notebook builds a neuron up one step at a time. Run each cell, read the note above it, and watch a weighted sum turn into a single squished output. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — NumPy

A neuron does two things: a **weighted sum** of its inputs (plus a bias), then an **activation** that squishes the result. We build it in two steps: (1) compute the pre-activation `z`, (2) pass it through the sigmoid.

### Step 1 — Compute the weighted sum z

Each input is multiplied by its own weight and the products are summed — that's exactly the dot product `w · x`. We then add the **bias** `b`, a single number that shifts the result up or down regardless of the inputs. The result `z` is called the pre-activation.

In [ ]:
import numpy as np

# one neuron: z = w . x + b, then sigmoid
w = np.array([0.5, -1.0, 2.0])   # one weight per input
b = 3.0                          # bias: shifts the result
x = np.array([4.0, 1.0, 2.0])    # the input vector

z = w @ x + b                    # dot product, then add bias
print("z =", z)                  # pre-activation

### Step 2 — Squish z through the sigmoid

The sigmoid `1 / (1 + e^(-z))` maps any real number into the open interval `(0, 1)`, so the neuron's output `a` reads like a soft yes/no. Large positive `z` approaches 1; large negative `z` approaches 0.

In [ ]:
a = 1.0 / (1.0 + np.exp(-z))     # sigmoid squishes z into (0, 1)
print("a =", round(float(a), 4)) # neuron output

## Visualize the data & results

_In a real digit classifier, how do one neuron's weighted pixel inputs add up to its output?_

### Step 1 — Train a small network and pick one neuron

We train a network with 16 hidden neurons on the real handwritten digits, with pixels scaled to 0–1. Then we pull out the weights and bias of a single hidden neuron (neuron 0). These 64 weights are exactly the `w` and `b` from Step 1 above, but learned from data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()                 # 1797 real 8x8 handwritten digits
X = digits.data / 16.0                  # pixel intensities scaled to 0..1

net = MLPClassifier(hidden_layer_sizes=(16,), max_iter=300,
                    random_state=0, alpha=1e-3).fit(X, digits.target)

W = net.coefs_[0][:, 0]                 # weights into hidden neuron 0
b = net.intercepts_[0][0]               # its bias

### Step 2 — Feed one digit through that neuron

We take one real image (a handwritten 0) and multiply each pixel by its weight to get a per-pixel contribution. Summing all of those and adding the bias gives `z`, then the sigmoid gives the output `a` — the same neuron computation as before, now over 64 pixel inputs.

In [ ]:
img = X[0]                              # one real digit image (a 0)

contrib = img * W                       # per-pixel weighted inputs
top = np.argsort(np.abs(contrib))[::-1][:3]   # 3 strongest pixels
z = float(contrib.sum() + b)            # full weighted sum + bias
a = 1.0 / (1.0 + np.exp(-z))            # sigmoid output

### Step 3 — Plot how the pieces add up to the output

The bar chart lines up the three strongest pixel contributions, the bias, the total `z`, and the final output `a`. It makes the neuron's whole story visible: a handful of weighted pixels (some pushing up, some down) plus a bias accumulate into `z`, which the sigmoid then squishes into the output.

In [ ]:
labels = ["pixel 18 * w", "pixel 26 * w", "pixel 12 * w", "bias", "z (sum)", "output a"]
values = list(contrib[top]) + [b, z, a]
colors = ["#ff7b72", "#ff7b72", "#4ea1ff", "#c89bff", "#ffb454", "#7ee787"]

plt.bar(labels, values, color=colors)
plt.title("One hidden neuron of a digit classifier: weighted pixels add to z, then sigmoid")
plt.axhline(0, color="gray", linewidth=0.8)
plt.show()